# HCP Subsampling Analysis — Difference Maps

**Purpose:** Compute voxel-wise difference maps between full/reference protocols and subsampled protocols  
**Data:** HCP subject 100408, 3T multi-shell DWI  
**Protocols:** 18 total — 3 shells (b1000, b2000, b3000) x 6 direction counts (20, 30, 45, 60, 75, 90)  
**Maps computed:**
- DTI: FA, MD (reference = per-shell n90)
- NODDI: NDI, ODI, FWF (reference = per-shell n90)

**Difference definition:** `diff = reference - subsampled` (signed, masked to brain)  
**MAE:** Mean Absolute Error within brain mask  
**Subsampling SNR:** `mean(reference) / std(diff)` — analogous to tSNR; higher = closer to reference  


In [ ]:
#!/usr/bin/env python
"""
HCP subsampling difference map analysis
Author: Raiyun Kabir
Date: April 2026
"""

import os
import csv
import subprocess
import numpy as np
import nibabel as nib
from pathlib import Path
import time

print(f"NumPy:   {np.__version__}")
print(f"NiBabel: {nib.__version__}")

# Verify FSL is available
result = subprocess.run(['fslmaths'], capture_output=True)
print(f"FSL fslmaths: {'available' if result.returncode in [0,1] else 'NOT FOUND — check module load fsl'}")
result = subprocess.run(['fslstats'], capture_output=True)
print(f"FSL fslstats: {'available' if result.returncode in [0,1] else 'NOT FOUND — check module load fsl'}")


## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_ROOT = "/scratch/rkabir5/StarterCodes_Data/results"
HCP_ROOT     = f"{RESULTS_ROOT}/HCP"

DTI_DIR   = f"{HCP_ROOT}/dtifit"
NODDI_DIR = f"{HCP_ROOT}/noddi"

MASK_FILE = "/scratch/rkabir5/StarterCodes_Data/100408_3T_Diffusion_preproc/100408/T1w/Diffusion/nodif_brain_mask.nii.gz"

OUTPUT_ROOT  = f"{HCP_ROOT}/analysis/diff_maps"
OUTPUT_DTI   = f"{OUTPUT_ROOT}/dtifit"
OUTPUT_NODDI = f"{OUTPUT_ROOT}/noddi"
TEMP_DIR     = f"{OUTPUT_ROOT}/tmp"

SHELLS = [1000, 2000, 3000]
NDIRS  = [20, 30, 45, 60, 75, 90]

PROTOCOLS = [
    {'shell': shell, 'ndirs': ndirs, 'name': f'b{shell}_n{ndirs}'}
    for shell in SHELLS
    for ndirs in NDIRS
]

Path(OUTPUT_DTI).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_NODDI).mkdir(parents=True, exist_ok=True)
Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

print(f"Results root:  {RESULTS_ROOT}")
print(f"DTI input:     {DTI_DIR}")
print(f"NODDI input:   {NODDI_DIR}")
print(f"Output DTI:    {OUTPUT_DTI}")
print(f"Output NODDI:  {OUTPUT_NODDI}")
print(f"Temp dir:      {TEMP_DIR}")
print(f"Protocols:     {len(PROTOCOLS)} total ({len(SHELLS)} shells x {len(NDIRS)} dir counts)")


## Verify Inputs

In [ ]:
print("Checking mask...")
assert os.path.exists(MASK_FILE), f"MISSING: {MASK_FILE}"
print(f"  OK: {MASK_FILE}")

print("\nChecking DTI per-shell references (n90)...")
for shell in SHELLS:
    for metric in ['FA', 'MD']:
        f = f"{DTI_DIR}/b{shell}_n90/dti_{metric}.nii.gz"
        assert os.path.exists(f), f"MISSING: {f}"
        print(f"  OK: b{shell}_n90/dti_{metric}.nii.gz")

print("\nChecking NODDI per-shell references (n90)...")
for shell in SHELLS:
    for metric in ['NDI', 'ODI', 'FWF']:
        f = f"{NODDI_DIR}/noddi_b{shell}_n90_936706/fit_{metric}.nii.gz"
        assert os.path.exists(f), f"MISSING: {f}"
        print(f"  OK: noddi_b{shell}_n90_936706/fit_{metric}.nii.gz")

print("\nChecking all 18 subsampled DTI protocols...")
missing = []
for p in PROTOCOLS:
    for metric in ['FA', 'MD']:
        f = f"{DTI_DIR}/{p['name']}/dti_{metric}.nii.gz"
        if not os.path.exists(f):
            missing.append(f)
if missing:
    print(f"  WARNING: {len(missing)} missing DTI files:")
    for m in missing: print(f"    {m}")
else:
    print("  All 36 DTI protocol files found.")

print("\nChecking all 18 subsampled NODDI protocols...")
missing = []
for p in PROTOCOLS:
    for metric in ['NDI', 'ODI', 'FWF']:
        f = f"{NODDI_DIR}/noddi_{p['name']}_936706/fit_{metric}.nii.gz"
        if not os.path.exists(f):
            missing.append(f)
if missing:
    print(f"  WARNING: {len(missing)} missing NODDI files:")
    for m in missing: print(f"    {m}")
else:
    print("  All 54 NODDI protocol files found.")

print("\nInput verification complete.")


## Load Reference Maps

In [ ]:
# ---- Brain mask info (nibabel for display only) ----
print("Loading brain mask info...")
mask_img = nib.load(MASK_FILE)
mask_data = mask_img.get_fdata().astype(bool)
print(f"  Mask shape:   {mask_data.shape}")
print(f"  Brain voxels: {mask_data.sum():,}")
del mask_data  # not needed further — FSL uses the mask file directly


# ============================================================
# FSL helper functions
# All heavy lifting (subtract, abs, mean, std) delegated to FSL
# ============================================================

def fsl_sub_mas(ref, sub, mask, out):
    """fslmaths ref -sub sub -mas mask -out out"""
    subprocess.run(
        ['fslmaths', ref, '-sub', sub, '-mas', mask, out],
        check=True, capture_output=True
    )

def fsl_mean(img, mask):
    """fslstats img -k mask -m  →  mean within mask"""
    r = subprocess.run(
        ['fslstats', img, '-k', mask, '-m'],
        check=True, capture_output=True, text=True
    )
    return float(r.stdout.strip())

def fsl_std(img, mask):
    """fslstats img -k mask -s  →  std within mask"""
    r = subprocess.run(
        ['fslstats', img, '-k', mask, '-s'],
        check=True, capture_output=True, text=True
    )
    return float(r.stdout.strip())

def fsl_mae(img, mask):
    """MAE = mean(|img|) within mask.
    Uses fslmaths img -abs tmp, then fslstats tmp -k mask -m."""
    tmp = f"{TEMP_DIR}/_mae_tmp.nii.gz"
    subprocess.run(
        ['fslmaths', img, '-abs', tmp],
        check=True, capture_output=True
    )
    return fsl_mean(tmp, mask)


# ============================================================
# Precompute reference map means (for SNR numerator)
# 3 shells x 2 DTI metrics  = 6 fslstats calls
# 3 shells x 3 NODDI metrics = 9 fslstats calls
# ============================================================

print("\nPrecomputing DTI reference means (fslstats)...")
dti_ref_mean = {}
for shell in SHELLS:
    dti_ref_mean[shell] = {}
    for metric in ['FA', 'MD']:
        ref_path = f"{DTI_DIR}/b{shell}_n90/dti_{metric}.nii.gz"
        dti_ref_mean[shell][metric] = fsl_mean(ref_path, MASK_FILE)
        print(f"  b{shell}_n90 {metric} mean = {dti_ref_mean[shell][metric]:.5f}")

print("\nPrecomputing NODDI reference means (fslstats)...")
noddi_ref_mean = {}
for shell in SHELLS:
    noddi_ref_mean[shell] = {}
    for metric in ['NDI', 'ODI', 'FWF']:
        ref_path = f"{NODDI_DIR}/noddi_b{shell}_n90_936706/fit_{metric}.nii.gz"
        noddi_ref_mean[shell][metric] = fsl_mean(ref_path, MASK_FILE)
        print(f"  b{shell}_n90 {metric} mean = {noddi_ref_mean[shell][metric]:.5f}")

print("\nReference means precomputed.")


## DTI Difference Maps

`diff = ref_n90_same_shell - subsampled`  
Positive: subsampled underestimates. Negative: subsampled overestimates.  
**SNR** = `mean(ref[mask]) / std(diff[mask])` — higher is better.

In [ ]:
print("Computing DTI difference maps (fslmaths + fslstats)...")
print("=" * 70)

dti_mae = {}
dti_snr = {}
t0 = time.time()

for p in PROTOCOLS:
    name  = p['name']
    shell = p['shell']
    dti_mae[name] = {}
    dti_snr[name] = {}

    for metric in ['FA', 'MD']:
        ref_path = f"{DTI_DIR}/b{shell}_n90/dti_{metric}.nii.gz"
        sub_path = f"{DTI_DIR}/{name}/dti_{metric}.nii.gz"
        out_path = f"{OUTPUT_DTI}/diff_{metric}_{name}.nii.gz"

        # fslmaths ref -sub sub -mas mask → signed diff map saved directly
        fsl_sub_mas(ref_path, sub_path, MASK_FILE, out_path)

        # MAE: fslmaths diff -abs tmp; fslstats tmp -k mask -m
        dti_mae[name][metric] = fsl_mae(out_path, MASK_FILE)

        # SNR: mean(ref) / std(diff); std computed by fslstats
        std_diff = fsl_std(out_path, MASK_FILE)
        dti_snr[name][metric] = (
            dti_ref_mean[shell][metric] / std_diff
            if std_diff > 0 else np.inf
        )

    print(
        f"  {name:15s}  "
        f"FA  MAE={dti_mae[name]['FA']:.5f} SNR={dti_snr[name]['FA']:6.1f}  "
        f"MD  MAE={dti_mae[name]['MD']:.2e} SNR={dti_snr[name]['MD']:6.1f}"
    )

print("=" * 70)
print(f"Done. Elapsed: {time.time()-t0:.1f}s")
print(f"Saved 36 DTI difference maps to: {OUTPUT_DTI}")


## NODDI Difference Maps

`diff = noddi_b{shell}_n90 - subsampled` (per-shell n90 reference)  
Positive: subsampled underestimates. Negative: subsampled overestimates.  
**SNR** = `mean(ref[mask]) / std(diff[mask])`


In [ ]:
print("Computing NODDI difference maps (fslmaths + fslstats)...")
print("=" * 80)

noddi_mae = {}
noddi_snr = {}
t0 = time.time()

for p in PROTOCOLS:
    name  = p['name']
    shell = p['shell']
    noddi_mae[name] = {}
    noddi_snr[name] = {}

    for metric in ['NDI', 'ODI', 'FWF']:
        ref_path = f"{NODDI_DIR}/noddi_b{shell}_n90_936706/fit_{metric}.nii.gz"
        sub_path = f"{NODDI_DIR}/noddi_{name}_936706/fit_{metric}.nii.gz"
        out_path = f"{OUTPUT_NODDI}/diff_{metric}_{name}.nii.gz"

        # fslmaths ref -sub sub -mas mask → signed diff map saved directly
        fsl_sub_mas(ref_path, sub_path, MASK_FILE, out_path)

        # MAE: fslmaths diff -abs tmp; fslstats tmp -k mask -m
        noddi_mae[name][metric] = fsl_mae(out_path, MASK_FILE)

        # SNR: mean(ref) / std(diff)
        std_diff = fsl_std(out_path, MASK_FILE)
        noddi_snr[name][metric] = (
            noddi_ref_mean[shell][metric] / std_diff
            if std_diff > 0 else np.inf
        )

    print(
        f"  {name:15s}  "
        f"NDI MAE={noddi_mae[name]['NDI']:.5f} SNR={noddi_snr[name]['NDI']:6.1f}  "
        f"ODI MAE={noddi_mae[name]['ODI']:.5f} SNR={noddi_snr[name]['ODI']:6.1f}  "
        f"FWF MAE={noddi_mae[name]['FWF']:.5f} SNR={noddi_snr[name]['FWF']:6.1f}"
    )

print("=" * 80)
print(f"Done. Elapsed: {time.time()-t0:.1f}s")
print(f"Saved 54 NODDI difference maps to: {OUTPUT_NODDI}")


## Summary — Tables and CSV

In [ ]:
# ---- Build rows ----
rows = []
for shell in SHELLS:
    for ndirs in NDIRS:
        name = f'b{shell}_n{ndirs}'
        rows.append({
            'protocol': name,
            'shell':    shell,
            'ndirs':    ndirs,
            'FA_MAE':   dti_mae[name]['FA'],
            'FA_SNR':   dti_snr[name]['FA'],
            'MD_MAE':   dti_mae[name]['MD'],
            'MD_SNR':   dti_snr[name]['MD'],
            'NDI_MAE':  noddi_mae[name]['NDI'],
            'NDI_SNR':  noddi_snr[name]['NDI'],
            'ODI_MAE':  noddi_mae[name]['ODI'],
            'ODI_SNR':  noddi_snr[name]['ODI'],
            'FWF_MAE':  noddi_mae[name]['FWF'],
            'FWF_SNR':  noddi_snr[name]['FWF'],
        })

# ---- Save CSV ----
csv_file = f"{OUTPUT_ROOT}/hcp_mae_snr_summary.csv"
fieldnames = [
    'protocol', 'shell', 'ndirs',
    'FA_MAE', 'FA_SNR', 'MD_MAE', 'MD_SNR',
    'NDI_MAE', 'NDI_SNR', 'ODI_MAE', 'ODI_SNR', 'FWF_MAE', 'FWF_SNR',
]
with open(csv_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print(f"CSV saved: {csv_file}")

# ---- MAE table ----
print()
W = 83
print("=" * W)
print("HCP SUMMARY -- Mean Absolute Error (MAE) within brain mask")
print("=" * W)
print(f"{'Protocol':<15} {'FA_MAE':>8} {'MD_MAE(e-4)':>12} {'NDI_MAE':>9} {'ODI_MAE':>9} {'FWF_MAE':>9}")
print("-" * W)
for shell in SHELLS:
    for r in [x for x in rows if x['shell'] == shell]:
        print(
            f"{r['protocol']:<15} "
            f"{r['FA_MAE']:>8.5f} "
            f"{r['MD_MAE']*1e4:>12.4f} "
            f"{r['NDI_MAE']:>9.5f} "
            f"{r['ODI_MAE']:>9.5f} "
            f"{r['FWF_MAE']:>9.5f}"
        )
    print("-" * W)

# ---- SNR table ----
print()
print("=" * W)
print("HCP SUMMARY -- Subsampling SNR  [SNR = mean(ref) / std(diff)]")
print("  Higher SNR = subsampled protocol closer to reference")
print("  Ref: Jones DK (2004), MRM 51(4):807-815")
print("=" * W)
print(f"{'Protocol':<15} {'FA_SNR':>8} {'MD_SNR':>8} {'NDI_SNR':>9} {'ODI_SNR':>9} {'FWF_SNR':>9}")
print("-" * W)
for shell in SHELLS:
    for r in [x for x in rows if x['shell'] == shell]:
        print(
            f"{r['protocol']:<15} "
            f"{r['FA_SNR']:>8.1f} "
            f"{r['MD_SNR']:>8.1f} "
            f"{r['NDI_SNR']:>9.1f} "
            f"{r['ODI_SNR']:>9.1f} "
            f"{r['FWF_SNR']:>9.1f}"
        )
    print("-" * W)

# ---- File counts ----
print()
dti_count   = len(list(Path(OUTPUT_DTI).glob('*.nii.gz')))
noddi_count = len(list(Path(OUTPUT_NODDI).glob('*.nii.gz')))
print(f"Output location:       {OUTPUT_ROOT}")
print(f"DTI diff maps saved:   {dti_count}")
print(f"NODDI diff maps saved: {noddi_count}")
print(f"Total NIfTI files:     {dti_count + noddi_count}")
print(f"CSV summary:           hcp_mae_snr_summary.csv")

# ---- Cleanup temp dir ----
import shutil
shutil.rmtree(TEMP_DIR)
print(f"\nTemp directory cleaned up: {TEMP_DIR}")
